In [ ]:
import pandas as pd
DATA_PATH = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_WITH_PAGES_TO_TEXT_FINAL.csv"

In [ ]:
data = pd.read_csv(DATA_PATH)

In [ ]:
data

## 1) General Explonatariy Analysis: Check invoice page vs not invoice page

In [ ]:
from ast import literal_eval
all_invoice_pages = []
all_invoice_pages_attachment_ids = []
all_invoice_page_numbers = []

all_not_invoice_pages = []
all_not_invoice_pages_attachment_ids = []
all_not_invoice_page_numbers = []

for index, row in data.iterrows():
    attachment_id = row['attachment_id']
    pages_to_text_dict = literal_eval(row['pages_to_text_dict'])
    invoice_page_start = row['invoice_page_start']
    invoice_page_end = row['invoice_page_end']
    
    if pd.isna(invoice_page_start) or pd.isna(invoice_page_end):
        # not invoice page
        
        for page, text in pages_to_text_dict.items():
            all_not_invoice_pages.append(text)
            all_not_invoice_pages_attachment_ids.append(attachment_id)
            all_not_invoice_page_numbers.append(int(page))
        continue
    
    invoice_page_start = int(invoice_page_start)
    invoice_page_end = int(invoice_page_end)
    for page, text in pages_to_text_dict.items():
        page = int(page)
        if invoice_page_start <= page <= invoice_page_end:
            all_invoice_pages.append(text)
            all_invoice_pages_attachment_ids.append(attachment_id)
            all_invoice_page_numbers.append(page)
        else:
            all_not_invoice_pages.append(text)
            all_not_invoice_pages_attachment_ids.append(attachment_id)
            all_not_invoice_page_numbers.append(page)


In [ ]:
len(all_invoice_pages), len(all_not_invoice_pages)

kv cost and amount regexes

In [ ]:
import re

# Matches: digits, optional decimal (, or .) with 1-2 digits, optional space, then € or EUR
AMOUNT_PATTERN = re.compile(
    r"\d+(?:[.,]\d{1,2})?\s*(?:€|EUR\b)",
    re.IGNORECASE,
)

KV_PATTERN = re.compile(r"\bKV\s*\d+(?:/\d+)?\b")

def replace_amounts(text: str) -> str:
    return AMOUNT_PATTERN.sub("<AMOUNT>", text)

def replace_kv(text: str) -> str:
    return KV_PATTERN.sub("<KV_COST>", text)

In [ ]:
for i in range(len(all_invoice_pages)):
    all_invoice_pages[i] = replace_amounts(all_invoice_pages[i])
    all_invoice_pages[i] = replace_kv(all_invoice_pages[i])
    
for i in range(len(all_not_invoice_pages)):
    all_not_invoice_pages[i] = replace_amounts(all_not_invoice_pages[i])
    all_not_invoice_pages[i] = replace_kv(all_not_invoice_pages[i])

### Basic length / character stats

In [ ]:
import numpy as np

def basic_stats(pages):
    char_lens = [len(p) for p in pages]
    word_lens = [len(p.split()) for p in pages]
    line_lens = [p.count("\n") + 1 for p in pages]
    digit_ratio = [
        sum(c.isdigit() for c in p) / len(p) if len(p) > 0 else 0
        for p in pages
    ]
    upper_ratio = [
        sum(c.isupper() for c in p) / sum(c.isalpha() for c in p)
        if sum(c.isalpha() for c in p) > 0 else 0
        for p in pages
    ]
    return {
        "n_pages": len(pages),
        "char_len_mean": float(np.mean(char_lens)),
        "char_len_median": float(np.median(char_lens)),
        "char_len_std": float(np.std(char_lens)),
        "word_len_mean": float(np.mean(word_lens)),
        "word_len_median": float(np.median(word_lens)),
        "line_len_mean": float(np.mean(line_lens)),
        "digit_ratio_mean": float(np.mean(digit_ratio)),
        "upper_ratio_mean": float(np.mean(upper_ratio)),
    }

stats_df = pd.DataFrame({
    "invoice": basic_stats(all_invoice_pages),
    "not_invoice": basic_stats(all_not_invoice_pages),
})
stats_df

### Length distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist([len(p) for p in all_invoice_pages], bins=50, alpha=0.6, label="invoice", density=True)
axes[0].hist([len(p) for p in all_not_invoice_pages], bins=50, alpha=0.6, label="not_invoice", density=True)
axes[0].set_title("Char length per page")
axes[0].set_xlabel("chars"); axes[0].legend()

axes[1].hist([len(p.split()) for p in all_invoice_pages], bins=50, alpha=0.6, label="invoice", density=True)
axes[1].hist([len(p.split()) for p in all_not_invoice_pages], bins=50, alpha=0.6, label="not_invoice", density=True)
axes[1].set_title("Word count per page")
axes[1].set_xlabel("words"); axes[1].legend()

plt.tight_layout(); plt.show()

### Most discriminative words (log-odds, additive smoothing)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(
    max_features=5000,
    lowercase=False,
    token_pattern=r"(?u)<AMOUNT>|<KV_COST>|\b[a-zA-ZäöüÄÖÜß]{3,}\b",
)
X = vec.fit_transform(all_invoice_pages + all_not_invoice_pages)
vocab = np.array(vec.get_feature_names_out())

n_inv = len(all_invoice_pages)
inv_counts = np.asarray(X[:n_inv].sum(axis=0)).ravel()
not_counts = np.asarray(X[n_inv:].sum(axis=0)).ravel()

alpha = 1.0
p_inv = (inv_counts + alpha) / (inv_counts.sum() + alpha * len(vocab))
p_not = (not_counts + alpha) / (not_counts.sum() + alpha * len(vocab))
log_odds = np.log(p_inv) - np.log(p_not)

top_n = 100
inv_top = np.argsort(log_odds)[-top_n:][::-1]
not_top = np.argsort(log_odds)[:top_n]

top_df = pd.DataFrame({
    "invoice_word": vocab[inv_top],
    "invoice_log_odds": log_odds[inv_top],
    "not_invoice_word": vocab[not_top],
    "not_invoice_log_odds": log_odds[not_top],
})
top_df

In [ ]:
for tok in ["<AMOUNT>", "<KV_COST>"]:
    idx = np.where(vocab == tok)[0]
    if len(idx) == 0:
        print(f"{tok}: NOT in vocab")
    else:
        i = idx[0]
        print(f"{tok}: inv_count={inv_counts[i]}, not_count={not_counts[i]}, log_odds={log_odds[i]:.3f}")

### Keyword / pattern presence rate

In [ ]:
import re

KEYWORDS = {
    "KV_COST":    r"<KV_COST>",
    "AMOUNT":     r"<AMOUNT>",
    "rechnung":   r"\brechnung\b",
    "invoice":    r"\binvoice\b",
    "betrag":     r"\bbetrag\b",
    "iban":       r"\biban\b",
    "ust_vat":    r"\b(ust|vat|mwst)\b",
    "eur_symbol": r"€",
    "amount_eur": r"\d+[.,]\d{2}\s*(€|eur)\b",
    "date":       r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b",
}

def keyword_rates(pages):
    rates = {}
    for name, pat in KEYWORDS.items():
        rx = re.compile(pat, re.IGNORECASE)
        rates[name] = float(np.mean([bool(rx.search(p)) for p in pages]))
    return rates

kw_df = pd.DataFrame({
    "invoice":     keyword_rates(all_invoice_pages),
    "not_invoice": keyword_rates(all_not_invoice_pages),
})
kw_df["diff"] = kw_df["invoice"] - kw_df["not_invoice"]
kw_df.sort_values("diff", ascending=False)

In [ ]:
top_df

In [ ]:
invoice_log_odds = dict(zip(top_df["invoice_word"], top_df["invoice_log_odds"]))
not_invoice_log_odds = dict(zip(top_df["not_invoice_word"], top_df["not_invoice_log_odds"]))

In [ ]:
# make every key lowercase for easier matching
invoice_log_odds = {k.lower(): v for k, v in invoice_log_odds.items()}
not_invoice_log_odds = {k.lower(): v for k, v in not_invoice_log_odds.items()}

In [ ]:
import re

# Improved basic algorithm:
# - tokenize identically to the CountVectorizer (keeps <AMOUNT>, <KV_COST>, case-sensitive)
# - score against the FULL vocab log-odds, not just the top-200
# - drop very rare words (noisy estimates)
# - length-normalize by number of scoring tokens
# - add the class log-prior so the natural decision threshold is 0
_TOKEN_RE = re.compile(r"<AMOUNT>|<KV_COST>|\b[a-zA-ZäöüÄÖÜß]{3,}\b")

_MIN_TOTAL_COUNT = 5
_keep = (inv_counts + not_counts) >= _MIN_TOTAL_COUNT
word_log_odds = {w: lo for w, lo, k in zip(vocab, log_odds, _keep) if k}

# log(P(invoice) / P(not_invoice)) — uniform across pages
log_prior = float(np.log(n_inv / max(len(all_not_invoice_pages), 1)))


def basic_algorithm_invoice_score(page_text: str) -> float:
    tokens = _TOKEN_RE.findall(page_text)
    if not tokens:
        return log_prior
    s = 0.0
    n = 0
    for t in tokens:
        lo = word_log_odds.get(t)
        if lo is not None:
            s += lo
            n += 1
    if n == 0:
        return log_prior
    return log_prior + s / n


In [ ]:
scores_invoice = [basic_algorithm_invoice_score(p) for p in all_invoice_pages]
scores_not_invoice = [basic_algorithm_invoice_score(p) for p in all_not_invoice_pages]



In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=scores_invoice,
    name="invoice",
    nbinsx=50,
    histnorm="probability density",
    opacity=0.6,
    marker_color="#1f77b4",
))
fig.add_trace(go.Histogram(
    x=scores_not_invoice,
    name="not_invoice",
    nbinsx=50,
    histnorm="probability density",
    opacity=0.6,
    marker_color="#ff7f0e",
))

fig.update_layout(
    title="Basic algorithm invoice scores",
    xaxis_title="score",
    yaxis_title="density",
    barmode="overlay",
    template="plotly_white",
    legend=dict(title="class"),
    hovermode="x unified",
)
fig.show()

In [ ]:
np.mean(scores_invoice), np.mean(scores_not_invoice)

In [ ]:
np.median(scores_invoice), np.median(scores_not_invoice)

In [ ]:
np.std(scores_invoice), np.std(scores_not_invoice)

In [ ]:
np.min(scores_invoice), np.max(scores_not_invoice)

In [ ]:
# print scores_not_invoice with score higher than 100
idx = [(i, s) for i, s in enumerate(scores_not_invoice) if s > 0]
print(idx)


In [ ]:
attachment_ids_with_high_not_invoice_score = [all_not_invoice_pages_attachment_ids[i] for i, s in idx]
page_numbers_with_high_not_invoice_score = [all_not_invoice_page_numbers[i] for i, s in idx]
print(attachment_ids_with_high_not_invoice_score)

In [ ]:
for page_number, a_id in zip(page_numbers_with_high_not_invoice_score, attachment_ids_with_high_not_invoice_score):
    print(f"Attachment ID: {a_id}, Page Number: {page_number}")

In [ ]:
file_paths = data[data['attachment_id'].isin(attachment_ids_with_high_not_invoice_score)].local_file_path.to_list()



In [ ]:
for path, page in zip(file_paths, page_numbers_with_high_not_invoice_score):
    print(f"File: {path}, Page: {page}")

### Stronger baseline: logistic regression on the same features

Evaluated with `GroupKFold` on `attachment_id` to avoid leakage between pages of the same document. Reports PR-AUC (preferred under class imbalance) and ROC-AUC.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

y = np.array([1] * n_inv + [0] * (X.shape[0] - n_inv))
groups = np.array(
    all_invoice_pages_attachment_ids + all_not_invoice_pages_attachment_ids
)

clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, solver="liblinear")
cv = GroupKFold(n_splits=5)
proba = cross_val_predict(clf, X, y, groups=groups, cv=cv, method="predict_proba")[:, 1]

print(f"PR-AUC : {average_precision_score(y, proba):.4f}")
print(f"ROC-AUC: {roc_auc_score(y, proba):.4f}")
print(classification_report(y, (proba >= 0.5).astype(int),
                            target_names=["not_invoice", "invoice"], digits=3))


In [ ]:
# Apples-to-apples metric for the improved basic algorithm
basic_scores = np.array(
    [basic_algorithm_invoice_score(p) for p in all_invoice_pages + all_not_invoice_pages]
)
print(f"basic algorithm PR-AUC : {average_precision_score(y, basic_scores):.4f}")
print(f"basic algorithm ROC-AUC: {roc_auc_score(y, basic_scores):.4f}")
